# Anti Money Laudenring Study
---

The primary objective of this analysis is to develop a machine learning model capable of identifying potentially suspicious transactions or customers for AML purposes.

We aim to perform data preprocessing and feature engineering to create meaningful inputs for modeling and then train and evaluate a machine learning model to detect suspicious behavior. Finally, we will provide interpretable insights that can support AML analysts and compliance decision-making.

Follow bellow some references for this study:

#### Clustering 

[1] Chandola, Varun, Arindam Banerjee, and Vipin Kumar. "Anomaly detection: A survey." ACM computing surveys (CSUR) 41.3 (2009): 1-58. Access on: https://dl.acm.org/doi/abs/10.1145/1541880.1541882

[2] Knox, Edwin M., and Raymond T. Ng. "Algorithms for mining distancebased outliers in large datasets." Proceedings of the international conference on very large data bases. Citeseer, 1998. Access on: https://www.researchgate.net/publication/2346156_Algorithms_for_Mining_Distance-Based_Outliers_in_Large_Datasets

[3] Jensen, Rasmus Ingemann Tuffveson, and Alexandros Iosifidis. "Fighting money laundering with statistics and machine learning." Ieee Access 11 (2023): 8889-8903. Access on: https://ieeexplore.ieee.org/stamp/stamp.jsp?arnumber=10025710

#### Dimension reduction

[4] Bakry, Ahmed N., et al. "Combating financial crimes with unsupervised learning techniques: clustering and dimensionality reduction for anti-money laundering." arXiv preprint arXiv:2403.00777 (2024). Access on: https://arxiv.org/pdf/2403.00777


### Imports
---

In [1]:
# system
import os
import sys
ROOT_PATH = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(ROOT_PATH)

# utils
from datetime import date, timedelta
import yfinance as yf

#visualization
import plotly.express as px

# data manipulation
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
pd.set_option('display.max_columns', None)

# personalized modules
from src.utils.dataset import get_raw_transactions

2026-02-19 21:27:27.125 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-19 21:27:27.127 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-19 21:27:27.129 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
random_state = 42

### Load data
---

To get started, we are going to load data from IBM dataset available on Kaggle. You can get the data on the following link: https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml.
After downloading it, make sure that the data is available in the right path "src/data/HI-Small_Trans.csv"

In [3]:
data_path = f"{ROOT_PATH}\\src\\data\\HI-Small_Trans.csv"
transactions_df = get_raw_transactions(data_path)

In [4]:
transactions_df.head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:20,10,8000EBD30,10,8000EBD30,"3,697.34",US Dollar,"3,697.34",US Dollar,Reinvestment,0
1,2022/09/01 00:20,3208,8000F4580,1,8000F5340,0.01,US Dollar,0.01,US Dollar,Cheque,0
2,2022/09/01 00:00,3209,8000F4670,3209,8000F4670,"14,675.57",US Dollar,"14,675.57",US Dollar,Reinvestment,0
3,2022/09/01 00:02,12,8000F5030,12,8000F5030,"2,806.97",US Dollar,"2,806.97",US Dollar,Reinvestment,0
4,2022/09/01 00:06,10,8000F5200,10,8000F5200,"36,682.97",US Dollar,"36,682.97",US Dollar,Reinvestment,0


### Transformations
---

In this step, the goal is to transform the original transaction-level dataset into a customer-level dataset. Since the objective is to cluster customers and assign a risk score at the customer level, we must first aggregate transactional information into meaningful customer-based features.

To uniquely identify each customer, we define a composite ID based on the combination of account and bank, ensuring consistent identification across the dataset.

Additionally, because transactions may occur in different currencies, it is necessary to standardize all monetary values into a single reference currency. To ensure comparability across the entire customer base, all transaction amounts will be converted to USD. For greater consistency and accuracy, the conversion can be performed using the exchange rate corresponding to the transaction date.

In [5]:
transformed_df = transactions_df.copy()
transformed_df.columns = [col.strip().lower().replace(" ", "_") for col in transformed_df.columns]

# create unique customer identifier
transformed_df["sender_customer"] = transformed_df["account"].astype(str) + "_" + transformed_df["from_bank"].astype(str)
transformed_df["receiver_customer"] = transformed_df["account.1"].astype(str) + "_" + transformed_df["to_bank"].astype(str)

#remove transactions with destination equal to the origin
transformed_df = transformed_df[transformed_df["payment_format"] != "Reinvestment"]

transformed_df["timestamp"] = pd.to_datetime(transformed_df['timestamp'])

In [6]:
currency2tickers = {
    "us dollar": "usdusd",
    "euro": "eurusd",
    "swiss franc": "chfusd",
    "yuan": "cnyusd",
    "shekel": "ilsusd",
    "rupee": "inrusd",
    "uk pound": "gbpusd",
    "ruble": "rubusd",
    "yen": "jpyusd",
    "bitcoin": "btc-usd",
    "canadian dollar": "cadusd",
    "australian dollar": "audusd",
    "mexican peso": "mxnusd",
    "saudi riyal": "sarusd",
    "brazil real": "brlusd"
}
tickers2currency = {v: k for k, v in currency2tickers.items()}
tickers = [
    "EURUSD=X", "CHFUSD=X", "CNYUSD=X", "ILSUSD=X", "INRUSD=X",
    "GBPUSD=X", "RUBUSD=X", "JPYUSD=X", "CADUSD=X", "AUDUSD=X",
    "MXNUSD=X", "SARUSD=X", "BRLUSD=X", "BTC-USD"
]
start_date = date.today() - timedelta(days=7)
currency_conversion = yf.download(tickers=tickers, start=start_date)
currency_conversion = currency_conversion["Close"].reset_index()
currency_conversion.columns = [column.lower().replace("=x", '') for column in currency_conversion.columns]

currency_conversion = currency_conversion.dropna().iloc[0:1, :]
currency_conversion["usdusd"] = 1.0
currency_conversion = currency_conversion.melt(id_vars=["date"], var_name="currency", value_name="exchange_rate")
currency_conversion["currency_name"] = currency_conversion["currency"].apply(lambda x: tickers2currency.get(x, "unknown"))
currency_conversion = currency_conversion.drop(columns=["date"])

C:\Users\ferna\AppData\Local\Temp\ipykernel_21820\1770237953.py:25: FutureWarning: YF.download() has changed argument auto_adjust default to True
  currency_conversion = yf.download(tickers=tickers, start=start_date)
[*********************100%***********************]  14 of 14 completed


In [7]:
transformed_df["payment_currency"] = transformed_df["payment_currency"].str.lower().str.strip()
transformed_df=transformed_df.merge(currency_conversion, left_on="payment_currency", right_on="currency_name", how="left")
transformed_df["usd_amount_paid"] = transformed_df["amount_paid"] * transformed_df["exchange_rate"]

In [8]:
transformed_df['transaction_date'] = transformed_df["timestamp"].dt.date
transformed_df["transaction_date"] = pd.to_datetime(transformed_df["transaction_date"])

In [9]:
transformed_df = (
    transformed_df[["timestamp", "transaction_date", "sender_customer", "receiver_customer", "usd_amount_paid"]]
    .rename(columns={"usd_amount_paid": "amount"})    
)

In [10]:
transformed_df.head()

,timestamp,transaction_date,sender_customer,receiver_customer,amount
0,2022-09-01 00:20:00,2022-09-01,8000F4580_3208,8000F5340_1,0.01
1,2022-09-01 00:26:00,2022-09-01,8000EC280_12,8017BF800_2439,7.66
2,2022-09-01 00:21:00,2022-09-01,8000EDEC0_1,80AEF5310_211050,383.71
3,2022-09-01 00:04:00,2022-09-01,8000F4510_1,8011305D0_11813,9.82
4,2022-09-01 00:08:00,2022-09-01,8000F4FE0_1,812ED62E0_245335,4.01


### Feature Engineering
---
In this section, we build the customer-level features that will be used in the AML model. Since our objective is to assess customer risk, transactional data is aggregated into behavioral indicators that summarize activity patterns.

The features are organized into three main groups:

1. **Transaction Behavior**: overall activity metrics such as volume, counts, and average amounts.

2. **Time Variance**: features capturing changes and patterns over time.

3. **Ratio Metrics**: proportional indicators, such as inflow/outflow relationships and concentration measures.

This structure allows the model to capture both activity intensity and behavioral patterns.

#### Transaction behavior features
---

In [11]:
sent_df = transformed_df.assign(
    customer_id=transformed_df["sender_customer"],
    direction="sent",
    amount=transformed_df["amount"],
    tx_count=1
)[["customer_id", "transaction_date", "direction", "amount", "tx_count"]]

received_df = transformed_df.assign(
    customer_id=transformed_df["receiver_customer"],
    direction="received",
    amount=transformed_df["amount"],
    tx_count=1
)[["customer_id", "transaction_date", "direction", "amount", "tx_count"]]

simplified_transaction_df = pd.concat([sent_df, received_df], ignore_index=True)

In [12]:
simplified_transaction_df.head()

,customer_id,transaction_date,direction,amount,tx_count
0,8000F4580_3208,2022-09-01,sent,0.01,1
1,8000EC280_12,2022-09-01,sent,7.66,1
2,8000EDEC0_1,2022-09-01,sent,383.71,1
3,8000F4510_1,2022-09-01,sent,9.82,1
4,8000F4FE0_1,2022-09-01,sent,4.01,1


In [13]:
transaction_behavior_df = (
    simplified_transaction_df.groupby(by=["customer_id", "direction"], as_index=False)
    .agg(
        transaction_count=("amount", "count"),
        total_amount=("amount", "sum"),
        median_amount=("amount", "median"),
        std_amount=("amount", "std"),
        max_amount=("amount", "max"),
    )
    .reset_index(drop=True)
    .pivot(index="customer_id", columns="direction")
    .fillna(0)
)
transaction_behavior_df.columns = [
    f"{metric}_{direction}"
    for metric, direction in transaction_behavior_df.columns
]
transaction_behavior_df = transaction_behavior_df.reset_index()

In [14]:
transaction_behavior_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83"
1,1004286A8_70,653.00,"103,018.00","298,090.67","30,960,152,350.53",268.62,"1,016.73",692.33,"6,980,038.34","5,106.41","980,734,541.09"
2,1004286F0_70,108.00,"18,663.00","36,735.46","3,567,068,065.53",187.74,908.63,418.18,"2,864,598.76","2,113.86","164,980,244.09"
3,100428738_70,98.00,"13,756.00","41,132.84","1,691,253,849.53",208.38,717.77,622.04,"2,730,606.70","2,506.31","194,716,581.44"
4,100428780_70,117.00,"17,264.00","116,501.99","4,121,130,736.36",226.69,800.80,"4,562.06","4,746,552.57","35,136.45","299,377,208.17"


#### Time features
---

In [15]:
simplified_time_df = (
    simplified_transaction_df
    .groupby(by=["customer_id", "transaction_date", "direction"])
    .agg(
        total_amount=("amount", "sum"),
        transaction_count=("tx_count", "sum"),
    )
    .reset_index()
)

reference_date_df = (
    simplified_time_df
    .groupby(by="customer_id", as_index=False)
    .agg(
        reference_date=("transaction_date", "max")
    )
)

simplified_time_df = simplified_time_df.merge(reference_date_df, on="customer_id", how="left", validate="many_to_one")

In [16]:
simplified_time_df["days_from_ref"] = (simplified_time_df["reference_date"] - simplified_time_df["transaction_date"]).dt.days

In [17]:
bins = [-1, 7, 30, 90]
labels = ["7d", "30d", "90d"]

simplified_time_df["window"] = pd.cut(
    simplified_time_df["days_from_ref"],
    bins=bins,
    labels=labels
)

In [18]:
simplified_time_df["window"] = simplified_time_df["window"].astype(str)
time_behavior_df = (
    simplified_time_df
    .groupby(["customer_id", "direction", "window"], as_index=False)
    .agg(
        total_amount=("total_amount", "sum"),
        transaction_count=("transaction_count", "sum")
    )
    .pivot_table(
        index="customer_id",
        columns=["direction", "window"],
        values=["total_amount", "transaction_count"],
        fill_value=0
    )
)
time_behavior_df.columns = [f"{metric}_{direction}_{window}" for metric, direction, window in time_behavior_df.columns]
time_behavior_df = time_behavior_df.reset_index()

#### Ratio features
---

In [19]:
transaction_behavior_df

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83"
1,1004286A8_70,653.00,"103,018.00","298,090.67","30,960,152,350.53",268.62,"1,016.73",692.33,"6,980,038.34","5,106.41","980,734,541.09"
2,1004286F0_70,108.00,"18,663.00","36,735.46","3,567,068,065.53",187.74,908.63,418.18,"2,864,598.76","2,113.86","164,980,244.09"
3,100428738_70,98.00,"13,756.00","41,132.84","1,691,253,849.53",208.38,717.77,622.04,"2,730,606.70","2,506.31","194,716,581.44"
4,100428780_70,117.00,"17,264.00","116,501.99","4,121,130,736.36",226.69,800.80,"4,562.06","4,746,552.57","35,136.45","299,377,208.17"
...,...,...,...,...,...,...,...,...,...,...,...
425110,814965930_27995,23.00,2.00,"7,148,145.10","4,752.40","363,677.64","2,376.20","270,050.43","2,247.92","643,526.27","3,965.72"
425111,8149659D0_36868,24.00,24.00,"458,056.79","458,056.79","3,803.72","3,803.72","48,809.86","48,809.86","175,270.90","175,270.90"
425112,814965AB0_245748,14.00,14.00,"381,840.26","381,840.26","34,106.34","34,106.34","11,454.94","11,454.94","34,106.34","34,106.34"
425113,814965B00_249118,35.00,30.00,"52,509.13","28,099.90",576.19,576.19,"3,293.46","1,708.33","16,111.77","9,322.76"


In [20]:
transaction_behavior_df["sent_received_ratio"] = transaction_behavior_df["total_amount_sent"]/(transaction_behavior_df["total_amount_received"] + 1)
transaction_behavior_df["transaction_direction_ratio"] = transaction_behavior_df["transaction_count_sent"]/(transaction_behavior_df["transaction_count_received"] + 1)

In [21]:
time_behavior_df["total_amount_30d_ratio"] = time_behavior_df["total_amount_sent_30d"] / (time_behavior_df["total_amount_received_30d"] + 1e-6)
time_behavior_df["total_amount_7d_ratio"] = time_behavior_df["total_amount_sent_7d"] / (time_behavior_df["total_amount_received_7d"] + 1e-6)

time_behavior_df["transaction_count_30d_ratio"] = time_behavior_df["transaction_count_sent_30d"] / (time_behavior_df["transaction_count_received_30d"] + 1e-6)
time_behavior_df["transaction_count_7d_ratio"] = time_behavior_df["transaction_count_sent_7d"] / (time_behavior_df["transaction_count_received_7d"] + 1e-6)

In [22]:
time_behavior_df.head()

,customer_id,total_amount_received_30d,total_amount_received_7d,total_amount_sent_30d,total_amount_sent_7d,transaction_count_received_30d,transaction_count_received_7d,transaction_count_sent_30d,transaction_count_sent_7d,total_amount_30d_ratio,total_amount_7d_ratio,transaction_count_30d_ratio,transaction_count_7d_ratio
0,100428660_70,"239,797.49","239,797.30","29,040,214,053.34","23,722,077,156.41",545.00,539.00,"49,166.00","119,506.00","121,103.08","98,925.54",90.21,221.72
1,1004286A8_70,"149,045.38","149,045.29","18,080,080,014.99","12,880,072,335.54",328.00,325.00,"30,137.00","72,881.00","121,305.88","86,417.17",91.88,224.25
2,1004286F0_70,"18,367.73","18,367.73","1,700,487,570.28","1,866,580,495.25",55.00,53.00,"5,370.00","13,293.00","92,580.16","101,622.82",97.64,250.81
3,100428738_70,"20,566.43","20,566.41","1,038,460,379.95","652,793,469.58",53.00,45.00,"3,989.00","9,767.00","50,492.99","31,740.76",75.26,217.04
4,100428780_70,"58,251.00","58,250.99","2,494,635,608.64","1,626,495,127.73",61.00,56.00,"4,949.00","12,315.00","42,825.63","27,922.19",81.13,219.91


#### Join all features
---

In [23]:
transaction_behavior_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent,sent_received_ratio,transaction_direction_ratio
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83","110,014.08",155.46
1,1004286A8_70,653.00,"103,018.00","298,090.67","30,960,152,350.53",268.62,"1,016.73",692.33,"6,980,038.34","5,106.41","980,734,541.09","103,861.18",157.52
2,1004286F0_70,108.00,"18,663.00","36,735.46","3,567,068,065.53",187.74,908.63,418.18,"2,864,598.76","2,113.86","164,980,244.09","97,098.84",171.22
3,100428738_70,98.00,"13,756.00","41,132.84","1,691,253,849.53",208.38,717.77,622.04,"2,730,606.70","2,506.31","194,716,581.44","41,115.88",138.95
4,100428780_70,117.00,"17,264.00","116,501.99","4,121,130,736.36",226.69,800.80,"4,562.06","4,746,552.57","35,136.45","299,377,208.17","35,373.61",146.31


In [24]:
time_behavior_df.head()

,customer_id,total_amount_received_30d,total_amount_received_7d,total_amount_sent_30d,total_amount_sent_7d,transaction_count_received_30d,transaction_count_received_7d,transaction_count_sent_30d,transaction_count_sent_7d,total_amount_30d_ratio,total_amount_7d_ratio,transaction_count_30d_ratio,transaction_count_7d_ratio
0,100428660_70,"239,797.49","239,797.30","29,040,214,053.34","23,722,077,156.41",545.00,539.00,"49,166.00","119,506.00","121,103.08","98,925.54",90.21,221.72
1,1004286A8_70,"149,045.38","149,045.29","18,080,080,014.99","12,880,072,335.54",328.00,325.00,"30,137.00","72,881.00","121,305.88","86,417.17",91.88,224.25
2,1004286F0_70,"18,367.73","18,367.73","1,700,487,570.28","1,866,580,495.25",55.00,53.00,"5,370.00","13,293.00","92,580.16","101,622.82",97.64,250.81
3,100428738_70,"20,566.43","20,566.41","1,038,460,379.95","652,793,469.58",53.00,45.00,"3,989.00","9,767.00","50,492.99","31,740.76",75.26,217.04
4,100428780_70,"58,251.00","58,250.99","2,494,635,608.64","1,626,495,127.73",61.00,56.00,"4,949.00","12,315.00","42,825.63","27,922.19",81.13,219.91


In [25]:
feature_df = transaction_behavior_df.merge(time_behavior_df, on="customer_id", how="left", validate="1:1")

In [26]:
feature_df.head()

,customer_id,transaction_count_received,transaction_count_sent,total_amount_received,total_amount_sent,median_amount_received,median_amount_sent,std_amount_received,std_amount_sent,max_amount_received,max_amount_sent,sent_received_ratio,transaction_direction_ratio,total_amount_received_30d,total_amount_received_7d,total_amount_sent_30d,total_amount_sent_7d,transaction_count_received_30d,transaction_count_received_7d,transaction_count_sent_30d,transaction_count_sent_7d,total_amount_30d_ratio,total_amount_7d_ratio,transaction_count_30d_ratio,transaction_count_7d_ratio
0,100428660_70,"1,084.00","168,672.00","479,594.79","52,762,291,209.75",238.44,"1,061.85",637.65,"8,093,751.98","5,551.91","1,615,849,280.83","110,014.08",155.46,"239,797.49","239,797.30","29,040,214,053.34","23,722,077,156.41",545.00,539.00,"49,166.00","119,506.00","121,103.08","98,925.54",90.21,221.72
1,1004286A8_70,653.00,"103,018.00","298,090.67","30,960,152,350.53",268.62,"1,016.73",692.33,"6,980,038.34","5,106.41","980,734,541.09","103,861.18",157.52,"149,045.38","149,045.29","18,080,080,014.99","12,880,072,335.54",328.00,325.00,"30,137.00","72,881.00","121,305.88","86,417.17",91.88,224.25
2,1004286F0_70,108.00,"18,663.00","36,735.46","3,567,068,065.53",187.74,908.63,418.18,"2,864,598.76","2,113.86","164,980,244.09","97,098.84",171.22,"18,367.73","18,367.73","1,700,487,570.28","1,866,580,495.25",55.00,53.00,"5,370.00","13,293.00","92,580.16","101,622.82",97.64,250.81
3,100428738_70,98.00,"13,756.00","41,132.84","1,691,253,849.53",208.38,717.77,622.04,"2,730,606.70","2,506.31","194,716,581.44","41,115.88",138.95,"20,566.43","20,566.41","1,038,460,379.95","652,793,469.58",53.00,45.00,"3,989.00","9,767.00","50,492.99","31,740.76",75.26,217.04
4,100428780_70,117.00,"17,264.00","116,501.99","4,121,130,736.36",226.69,800.80,"4,562.06","4,746,552.57","35,136.45","299,377,208.17","35,373.61",146.31,"58,251.00","58,250.99","2,494,635,608.64","1,626,495,127.73",61.00,56.00,"4,949.00","12,315.00","42,825.63","27,922.19",81.13,219.91


In [27]:
save_path = f"{ROOT_PATH}\\src\\data\\preprocessed_customer_level_features.csv"
feature_df.to_csv(save_path, index=False)

#### Checking correlation
---

In [ ]:
#plot a correalation heatmap using plotly
correlation_columns = list(feature_df.columns)
correlation_columns.remove("customer_id")
corr_matrix = feature_df[correlation_columns].corr()

fig = px.imshow(
    corr_matrix,
    color_continuous_scale="RdBu",   # diverging scale (best for correlations)
    zmin=-1,
    zmax=1,
    aspect="auto",
    text_auto=".2f",
    title="Correlation Heatmap – Transaction Behavior Features"
)

fig.update_layout(
    width=1400,
    height=1000,
    coloraxis_colorbar=dict(
        title="Correlation",
        ticks="outside"
    )
)

fig.show()

In [ ]:
correlation_columns

In [ ]:
feature_df.shape